# Report Generator – Debug / Development Notebook

Primary orchestration notebook for the **Daily Detail Delivery Audit Report** job.
Use this for interactive development and debugging before running
the production wrapper (`src.jobs.report_generator.ReportGeneratorJob`).

**Environment**: boilerplate

## Cell 1 – Setup & Imports

In [1]:
import logging
logging.getLogger().handlers.clear()

import sys
import os
from datetime import datetime, timedelta

# Add project root for imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

script_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

from config.settings import load_config, set_config
from src.utils.helpers import setup_logging, setup_keyring

config = load_config(script_dir)
set_config(config)
setup_logging(config, script_dir)
setup_keyring()

logger = logging.getLogger('report_generator')
logger.info('Config loaded successfully')

Loading configuration for environment: dev (Developer: local.yaml)
2026-03-31 18:23:55,465 - root - INFO - Setup Logging initialized for environment
2026-03-31 18:23:55,970 - report_generator - INFO - Config loaded successfully


## Cell 2 – Parameters
Change these when debugging different schedule types.

In [2]:
# ┌──────────────────────────────────────────────────────────┐
# │  PARAMETERS – tweak these for local testing              │
# └──────────────────────────────────────────────────────────┘
SCHEDULE_TYPE  = 'daily'
# REFERENCE_DATE = None
REFERENCE_DATE = '2026-03-31'          # None = auto (today Jakarta), or 'YYYY-MM-DD'
OUTPUT_FORMAT  = config.get('REPORT', {}).get('OUTPUT_FORMAT', 'xlsx')   # xlsx / csv
JOB_CODE       = 'DAILY_DETAIL_DELIVERY_AUDIT'

logger.info(f'SCHEDULE_TYPE={SCHEDULE_TYPE}  OUTPUT_FORMAT={OUTPUT_FORMAT}  JOB_CODE={JOB_CODE}')

2026-03-31 18:24:02,632 - report_generator - INFO - SCHEDULE_TYPE=daily  OUTPUT_FORMAT=xlsx  JOB_CODE=DAILY_DETAIL_DELIVERY_AUDIT


## Cell 3 – Calculate Date Range

In [3]:
from src.utils.dateutils import calculate_date_range, generate_looping_dates, date_for_filename

start_date, end_date = calculate_date_range(SCHEDULE_TYPE, REFERENCE_DATE)
print(f'Date range: {start_date} -> {end_date}')

if SCHEDULE_TYPE == 'daily':
    date_ranges = [(start_date, end_date)]
else:
    date_ranges = generate_looping_dates(start_date, end_date)

print(f'Looping ranges: {len(date_ranges)}')
for i, (s, e) in enumerate(date_ranges[:5]):
    print(f'  [{i}] {s} -> {e}')
if len(date_ranges) > 5:
    print(f'  ... and {len(date_ranges) - 5} more')

2026-03-31 18:24:05,320 - src.utils.dateutils - INFO - [DAILY] date range: 2026-03-30 -> 2026-03-31
Date range: 2026-03-30 -> 2026-03-31
Looping ranges: 1
  [0] 2026-03-30 -> 2026-03-31


## Cell 3A – Fetch Region List

Query `analysis_services.tb_mst_area_by_province` to get the list of regions.
Each region maps to a OneDrive upload folder and a date stamp for filenames.

In [4]:
import psycopg2
from src.utils.query_loader import load_region_query, inject_parameters

manifest_path = config.get('QUERIES', {}).get('MANIFEST_PATH', 'src/sql/manifest.json')
manifest_full = os.path.join(script_dir, manifest_path)

# Load region query from manifest
region_sql = load_region_query(manifest_full)

if not region_sql:
    print('No region_query defined in manifest – skipping region fetch')
    regions = []
else:
    # Open DB connection for region query
    conn = psycopg2.connect(
        host=config.get('DB_HOST', '127.0.0.1'),
        port=config.get('DB_PORT', 5432),
        dbname=config.get('DB_NAME', 'your_db'),
        user=config.get('DB_USER', 'your_user'),
        password=config.get('DB_PASSWORD', 'your_password'),
    )
    logger.info('DB connection established')

    cur = conn.cursor()
    try:
        cur.execute(region_sql)
        columns = [d[0] for d in cur.description]
        regions = [dict(zip(columns, row)) for row in cur.fetchall()]
    finally:
        cur.close()
        conn.commit()

    print(f'Found {len(regions)} region(s):')
    for r in regions:
        print(f"  {r['region']} -> {r['region2']} | upload: {r['upload_url']} | tgl: {r['tgl']}")

2026-03-31 18:24:09,159 - src.utils.query_loader - INFO - Region query loaded: regions -> query_regions.sql
2026-03-31 18:24:09,313 - report_generator - INFO - DB connection established
Found 6 region(s):
  JABAR -> JABAR | upload: https://graph.microsoft.com/v1.0/me/drive/root:/Projects/Report_Audit/Delivery/JABAR | tgl: 20260330
  JABODETABEK -> JABODETABEK | upload: https://graph.microsoft.com/v1.0/me/drive/root:/Projects/Report_Audit/Delivery/JABODETABEK | tgl: 20260330
  JATENG DIY -> JATENG_DIY | upload: https://graph.microsoft.com/v1.0/me/drive/root:/Projects/Report_Audit/Delivery/JATENG_DIY | tgl: 20260330
  JATIM BALI NUSRA -> JATIM_BALI_NUSRA | upload: https://graph.microsoft.com/v1.0/me/drive/root:/Projects/Report_Audit/Delivery/JATIM_BALI_NUSRA | tgl: 20260330
  KAL SULAM PAPUA -> KAL_SULAM_PAPUA | upload: https://graph.microsoft.com/v1.0/me/drive/root:/Projects/Report_Audit/Delivery/KAL_SULAM_PAPUA | tgl: 20260330
  SUMATERA -> SUMATERA | upload: https://graph.microsoft.co

## Cell 4 – Load SQL Queries from Manifest

In [5]:
from src.utils.query_loader import load_all_queries, load_manifest_meta, validate_manifest

manifest_path = config.get('QUERIES', {}).get('MANIFEST_PATH', 'src/sql/manifest.json')
manifest_full = os.path.join(script_dir, manifest_path)

validate_manifest(manifest_full)

# Load template metadata
manifest_meta = load_manifest_meta(manifest_full)
template_file = manifest_meta.get('template_file')
template_header_rows = manifest_meta.get('template_header_rows', 1)
print(f'Template file: {template_file}')
print(f'Template header rows: {template_header_rows}')

all_queries = load_all_queries(manifest_full)
for q in all_queries:
    print(f"  Query: {q['name']} -> sheet: {q['sheet_name']} ({q['file']})")
    print(f"    SQL preview: {q['sql'][:120]}...")

2026-03-31 18:24:14,989 - src.utils.query_loader - INFO - Manifest loaded: 1 queries from c:\ETL as Code\svc-delivery-region-audit-report\src/sql/manifest.json
2026-03-31 18:24:14,990 - src.utils.query_loader - INFO -   [OK] delivery_audit -> query_delivery_audit.sql (sheet: Delivery_Audit)
Template file: None
Template header rows: 1
2026-03-31 18:24:14,992 - src.utils.query_loader - INFO - Manifest loaded: 1 queries from c:\ETL as Code\svc-delivery-region-audit-report\src/sql/manifest.json
2026-03-31 18:24:14,993 - src.utils.query_loader - INFO - Loaded 1 SQL queries from manifest
  Query: delivery_audit -> sheet: Delivery_Audit (query_delivery_audit.sql)
    SQL preview: SET max_parallel_workers_per_gather = 0;

SELECT aws.order_time_dt,
       cmdm.full_name,
       cmdm.channel,
       a...


## Cell 5 – Loop Per Region: Build Queries, Export, Upload

For each region: inject parameters → export Excel → upload to OneDrive → collect link.

In [6]:
import pandas as pd
import xlsxwriter
from datetime import datetime, date
from src.utils.excel_exporter import export_report
from src.utils.onedrive_uploader import upload_file, create_share_link
from src.utils.query_loader import load_awb_query, load_batch_size

file_path_base = config.get('REPORT', {}).get('FILE_PATH', './data/reports')
os.makedirs(os.path.join(script_dir, file_path_base), exist_ok=True)

ext = 'xlsx' if OUTPUT_FORMAT == 'xlsx' else 'csv'
region_files = []

# Load AWB batching config from manifest
awb_sql_raw = load_awb_query(manifest_full)
BATCH_SIZE = load_batch_size(manifest_full)
print(f'Batching enabled: {awb_sql_raw is not None} | batch_size={BATCH_SIZE}')

for idx, region_row in enumerate(regions, 1):
    region  = region_row['region']
    region2 = region_row['region2']
    upload_url = region_row['upload_url']
    tgl     = region_row['tgl']

    print(f'\n═══ [{idx}/{len(regions)}] Region: {region} ({region2}) ═══')

    # ── 5a. Inject parameters ────────────────────────────────────
    params = {'STARTDATE': start_date, 'ENDDATE': end_date, 'region': region}

    filename = f'Daily_Report_Delivery_By_{region2}_{tgl}'
    output_file = os.path.join(script_dir, file_path_base, f'{filename}.{ext}')

    if awb_sql_raw:
        # ── BATCHED MODE ─────────────────────────────────────────
        # Phase 1: Lightweight AWB fetch
        awb_sql = inject_parameters(awb_sql_raw, params)
        cur = conn.cursor()
        cur.execute(awb_sql)
        awb_list = [row[0] for row in cur.fetchall()]
        cur.close()

        total_batches = (len(awb_list) + BATCH_SIZE - 1) // BATCH_SIZE
        print(f'  Found {len(awb_list):,} AWB → {total_batches} batches of {BATCH_SIZE:,}')

        # Phase 2: Batch loop
        base_query = inject_parameters(all_queries[0]['sql'], params)
        sheet_name = all_queries[0]['sheet_name']
        all_results = []

        for batch_num, i in enumerate(range(0, len(awb_list), BATCH_SIZE), 1):
            batch_awb = awb_list[i:i+BATCH_SIZE]
            batch_query = base_query + "\n  AND aws.awb = ANY(%s)"
            batch_df = pd.read_sql(batch_query, conn, params=(batch_awb,))
            all_results.append(batch_df)
            print(f'    Batch {batch_num}/{total_batches}: {len(batch_awb):,} AWB → {len(batch_df):,} rows')

        # Phase 3: Combine + export
        if all_results:
            combined_df = pd.concat(all_results, ignore_index=True)
            print(f'  Total: {len(combined_df):,} rows collected')

            with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
                combined_df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
                ws = writer.sheets[sheet_name[:31]]
                for col_num, col_name in enumerate(combined_df.columns):
                    col_max = combined_df[col_name].astype(str).str.len().max()
                    col_max = 0 if pd.isna(col_max) else int(col_max)
                    col_max = max(col_max, len(str(col_name)))
                    ws.set_column(col_num, col_num, min(col_max + 2, 50))

            result_path = os.path.abspath(output_file)
        else:
            print(f'  ⚠️ No AWB found for {region}')
            continue
    else:
        # ── ORIGINAL MODE (no batching) ──────────────────────────
        sheets_config = []
        for q in all_queries:
            rendered_sql = inject_parameters(q['sql'], params)
            sheets_config.append((q['sheet_name'], rendered_sql))

        result_path = export_report(
            output_path=output_file,
            sheets_config=sheets_config,
            conn=conn,
            output_format=OUTPUT_FORMAT,
            append=False,
            format_config={'header_bold': True, 'auto_width': True},
        )

    print(f'  Exported → {result_path}')

    # ── 5d. Upload to OneDrive (region folder) ───────────────────
    try:
        upload_result = upload_file(
            file_path=result_path,
            remote_folder=upload_url,
            conn=conn,
            config=config,
        )
        file_link = upload_result.get('webUrl', '')
        item_id = upload_result.get('id')
        if item_id:
            try:
                file_link = create_share_link(item_id=item_id, conn=conn, config=config)
            except Exception as e:
                logger.warning(f'Could not create share link: {e}')

        region_files.append({
            'region': region,
            'filename': f'{filename}.{ext}',
            'link': file_link,
        })
        print(f'  Uploaded → {file_link}')
    except Exception as e:
        logger.error(f'Upload failed for {region}: {e}')
        print(f'  ⚠️ Upload failed: {e}')
        region_files.append({
            'region': region,
            'filename': f'{filename}.{ext}',
            'link': '',
        })

print(f'\n✅ All regions processed: {len(region_files)} files')
for rf in region_files:
    print(f"  {rf['region']}: {rf['filename']} → {rf['link'][:60] if rf['link'] else 'ERROR'}...")

2026-03-31 18:24:19,793 - src.utils.query_loader - INFO - AWB query loaded: awb_list -> query_awb_list.sql
Batching enabled: True | batch_size=3000

═══ [1/6] Region: JABAR (JABAR) ═══
  Found 36,754 AWB → 13 batches of 3,000


C:\Users\ArifSetiadi\AppData\Local\Temp\ipykernel_28272\1409596087.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  batch_df = pd.read_sql(batch_query, conn, params=(batch_awb,))


    Batch 1/13: 3,000 AWB → 3,000 rows
    Batch 2/13: 3,000 AWB → 3,000 rows
    Batch 3/13: 3,000 AWB → 3,000 rows
    Batch 4/13: 3,000 AWB → 3,000 rows
    Batch 5/13: 3,000 AWB → 3,000 rows
    Batch 6/13: 3,000 AWB → 3,000 rows
    Batch 7/13: 3,000 AWB → 3,000 rows
    Batch 8/13: 3,000 AWB → 3,000 rows
    Batch 9/13: 3,000 AWB → 3,000 rows
    Batch 10/13: 3,000 AWB → 3,000 rows
    Batch 11/13: 3,000 AWB → 3,000 rows
    Batch 12/13: 3,000 AWB → 3,000 rows
    Batch 13/13: 754 AWB → 754 rows
  Total: 36,754 rows collected
  Exported → c:\ETL as Code\svc-delivery-region-audit-report\data\reports\Daily_Report_Delivery_By_JABAR_20260330.xlsx
2026-03-31 18:30:11,009 - src.utils.onedrive_uploader - INFO - OAuth credentials loaded from DB: config_upload_onedrive_finance
2026-03-31 18:30:11,033 - src.utils.onedrive_uploader - INFO - Uploading Daily_Report_Delivery_By_JABAR_20260330.xlsx (8,858,048 bytes) -> https://graph.microsoft.com/v1.0/me/drive/root:/Projects/Report_Audit/Delive

C:\Users\ArifSetiadi\AppData\Local\Temp\ipykernel_28272\1409596087.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  batch_df = pd.read_sql(batch_query, conn, params=(batch_awb,))


    Batch 1/78: 3,000 AWB → 3,000 rows
    Batch 2/78: 3,000 AWB → 3,000 rows
    Batch 3/78: 3,000 AWB → 3,000 rows
    Batch 4/78: 3,000 AWB → 3,000 rows
    Batch 5/78: 3,000 AWB → 3,000 rows
    Batch 6/78: 3,000 AWB → 3,000 rows
    Batch 7/78: 3,000 AWB → 3,000 rows
    Batch 8/78: 3,000 AWB → 3,000 rows
    Batch 9/78: 3,000 AWB → 3,000 rows
    Batch 10/78: 3,000 AWB → 3,000 rows
    Batch 11/78: 3,000 AWB → 3,000 rows
    Batch 12/78: 3,000 AWB → 3,000 rows
    Batch 13/78: 3,000 AWB → 3,000 rows
    Batch 14/78: 3,000 AWB → 3,000 rows
    Batch 15/78: 3,000 AWB → 3,000 rows
    Batch 16/78: 3,000 AWB → 3,000 rows
    Batch 17/78: 3,000 AWB → 3,000 rows
    Batch 18/78: 3,000 AWB → 3,000 rows
    Batch 19/78: 3,000 AWB → 3,000 rows
    Batch 20/78: 3,000 AWB → 3,000 rows
    Batch 21/78: 3,000 AWB → 3,000 rows
    Batch 22/78: 3,000 AWB → 3,000 rows
    Batch 23/78: 3,000 AWB → 3,000 rows
    Batch 24/78: 3,000 AWB → 3,000 rows
    Batch 25/78: 3,000 AWB → 3,000 rows
    Batch

...

    Batch 1/29: 3,000 AWB → 3,000 rows
    Batch 2/29: 3,000 AWB → 3,000 rows
    Batch 3/29: 3,000 AWB → 3,000 rows
    Batch 4/29: 3,000 AWB → 3,000 rows
    Batch 5/29: 3,000 AWB → 3,000 rows
    Batch 6/29: 3,000 AWB → 3,000 rows
    Batch 7/29: 3,000 AWB → 3,000 rows
    Batch 8/29: 3,000 AWB → 3,000 rows
    Batch 9/29: 3,000 AWB → 3,000 rows
    Batch 10/29: 3,000 AWB → 3,000 rows
    Batch 11/29: 3,000 AWB → 3,000 rows
    Batch 12/29: 3,000 AWB → 3,000 rows
    Batch 13/29: 3,000 AWB → 3,000 rows
    Batch 14/29: 3,000 AWB → 3,000 rows
    Batch 15/29: 3,000 AWB → 3,000 rows
    Batch 16/29: 3,000 AWB → 3,000 rows
    Batch 17/29: 3,000 AWB → 3,000 rows
    Batch 18/29: 3,000 AWB → 3,000 rows
    Batch 19/29: 3,000 AWB → 3,000 rows
    Batch 20/29: 3,000 AWB → 3,000 rows
    Batch 21/29: 3,000 AWB → 3,000 rows
    Batch 22/29: 3,000 AWB → 3,000 rows
    Batch 23/29: 3,000 AWB → 3,000 rows
    Batch 24/29: 3,000 AWB → 3,000 rows
    Batch 25/29: 3,000 AWB → 3,000 rows
    Batch

C:\Users\ArifSetiadi\AppData\Local\Temp\ipykernel_28272\1409596087.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  batch_df = pd.read_sql(batch_query, conn, params=(batch_awb,))


    Batch 1/4: 3,000 AWB → 3,000 rows
    Batch 2/4: 3,000 AWB → 3,000 rows
    Batch 3/4: 3,000 AWB → 3,000 rows
    Batch 4/4: 1,724 AWB → 1,724 rows
  Total: 10,724 rows collected
  Exported → c:\ETL as Code\svc-delivery-region-audit-report\data\reports\Daily_Report_Delivery_By_KAL_SULAM_PAPUA_20260330.xlsx
2026-03-31 19:05:51,402 - src.utils.onedrive_uploader - INFO - OAuth credentials loaded from DB: config_upload_onedrive_finance
2026-03-31 19:05:51,429 - src.utils.onedrive_uploader - INFO - Uploading Daily_Report_Delivery_By_KAL_SULAM_PAPUA_20260330.xlsx (2,750,096 bytes) -> https://graph.microsoft.com/v1.0/me/drive/root:/Projects/Report_Audit/Delivery/KAL_SULAM_PAPUA/Daily_Report_Delivery_By_KAL_SULAM_PAPUA_20260330.xlsx
2026-03-31 19:05:52,530 - report_generator - ERROR - Upload failed for KAL SULAM PAPUA: 400 Client Error: Bad Request for url: https://graph.microsoft.com/v1.0/drives/b!g9DfpjQpq0G0J_MeEmnGZnfcRyAFHPlAuurHGlkiSuCQ9M5Vrr8cQra2tl6FhfoZ/root:/https://graph.microso

C:\Users\ArifSetiadi\AppData\Local\Temp\ipykernel_28272\1409596087.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  batch_df = pd.read_sql(batch_query, conn, params=(batch_awb,))


    Batch 1/8: 3,000 AWB → 3,000 rows
    Batch 2/8: 3,000 AWB → 3,000 rows
    Batch 3/8: 3,000 AWB → 3,000 rows
    Batch 4/8: 3,000 AWB → 3,000 rows
    Batch 5/8: 3,000 AWB → 3,000 rows
    Batch 6/8: 3,000 AWB → 3,000 rows
    Batch 7/8: 3,000 AWB → 3,000 rows
    Batch 8/8: 1,125 AWB → 1,125 rows
  Total: 22,125 rows collected
  Exported → c:\ETL as Code\svc-delivery-region-audit-report\data\reports\Daily_Report_Delivery_By_SUMATERA_20260330.xlsx
2026-03-31 19:07:37,933 - src.utils.onedrive_uploader - INFO - OAuth credentials loaded from DB: config_upload_onedrive_finance
2026-03-31 19:07:37,956 - src.utils.onedrive_uploader - INFO - Uploading Daily_Report_Delivery_By_SUMATERA_20260330.xlsx (5,510,693 bytes) -> https://graph.microsoft.com/v1.0/me/drive/root:/Projects/Report_Audit/Delivery/SUMATERA/Daily_Report_Delivery_By_SUMATERA_20260330.xlsx
2026-03-31 19:07:38,077 - report_generator - ERROR - Upload failed for SUMATERA: 400 Client Error: Bad Request for url: https://graph.mic

## Cell 6 – (Merged into Cell 5 – Region Loop handles Export)

Export is now handled inside the region loop above. This cell is kept as placeholder.

In [ ]:
# Export + Upload handled in Cell 5 (region loop)
# This cell intentionally left empty
print('Export + Upload already completed in Cell 5 (region loop)')

## Cell 7 – (Merged into Cell 5 – Region Loop handles Upload)

Upload is now handled inside the region loop above. This cell is kept as placeholder.

In [ ]:
# Upload handled in Cell 5 (region loop)
# This cell intentionally left empty
print('Upload already completed in Cell 5 (region loop)')

## Cell 8 – Send Email Notification (Single Email with All Region Links)

In [ ]:
from src.utils.email_notifier import send_report_email
from src.db.repository.oauth_repository import get_smtp_config
from src.utils.dateutils import format_date, now_jakarta

# Fetch SMTP config + email recipients from DB
smtp_cfg = get_smtp_config(conn, 'DAILY_DETAIL_DELIVERY_AUDIT')
print(f'SMTP config: {smtp_cfg}')

if smtp_cfg and smtp_cfg.get('email'):
    try:
        to_email   = smtp_cfg['email']
        cc_email   = smtp_cfg.get('cc_email')
        bcc_email  = smtp_cfg.get('bcc_email')
        from_email = 'ANTERAJA INFO <internal.info@anteraja.id>'
        smtp_host  = smtp_cfg['host']
        smtp_port  = smtp_cfg.get('port', 587)
        username   = smtp_cfg['username']
        password   = smtp_cfg['password']

        template_path = os.path.join(
            script_dir, config.get('EMAIL', {}).get('TEMPLATE_PATH', 'config/config/email_template.jinja2')
        )

        subject = f"Anteraja Daily Detail Delivery Audit Report ({format_date(start_date, '%d-%m-%Y')})"

        context = {
            'report_title':   'Daily Detail Delivery Audit Report',
            'schedule_type':  SCHEDULE_TYPE,
            'report_date':    format_date(start_date, '%d %B %Y'),
            'start_date':     start_date,
            'end_date':       end_date,
            'generated_at':   now_jakarta().strftime('%Y-%m-%d %H:%M:%S'),
            'region_files':   region_files,
        }

        send_report_email(
            template_path=template_path,
            to_email=to_email,
            cc_email=cc_email,
            bcc_email=bcc_email,
            subject=subject,
            context=context,
            from_email=from_email,
            smtp_host=smtp_host,
            smtp_port=smtp_port,
            username=username,
            password=password,
        )
        print(f'✅ Email sent to {to_email}')
    except Exception as e:
        logger.warning(f'Email error: {e}')
        print(f'⚠️  Email skipped: {e}')
else:
    print('⚠️  No SMTP config found in DB – skipping notification')

## Cell 9 – Cleanup

In [ ]:
if conn and not conn.closed:
    conn.close()
    logger.info('DB connection closed')

print('🏁 Report generator notebook completed')